# Model C: Prescriptive AI (Actionable Suggestions)

Simply returning a classification like `DECEASED_DISBURSEMENT` leaves the District Finance Officer wondering what to do.

Here, we pull real data from `TS-PS4-1.csv` and `TS-PS4-2.csv`, find a mathematical anomaly (a transaction occurring after a death date), and feed that raw data into an Expert System / LLM simulation to generate an actionable English suggestion.


## 1. Extract a Real Case from the Datasets


In [7]:
import pandas as pd

# Load datasets
transactions = pd.read_csv('../data/TS-PS4-1.csv')
deaths = pd.read_csv('../data/TS-PS4-2.csv')

# Ensure dates are datetime objects
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'])
deaths['death_date'] = pd.to_datetime(deaths['death_date'])

# Perform the deterministic join
merged = pd.merge(transactions, deaths, on='aadhaar', suffixes=('_txn', '_registry'))

# Find transactions that occurred AFTER the official death date
flagged_deceased = merged[merged['transaction_date'] > merged['death_date']]

if not flagged_deceased.empty:
    # Take the first real flagged case
    real_case = flagged_deceased.iloc[0]
    
    case_context = {
        "flag_type": "DECEASED_DISBURSEMENT",
        "risk_score": 100,
        "beneficiary_id": real_case['beneficiary_id'],
        "aadhaar": real_case['aadhaar'],
        "transaction_date": real_case['transaction_date'].strftime('%Y-%m-%d'),
        "death_date": real_case['death_date'].strftime('%Y-%m-%d'),
        "days_post_mortem": (real_case['transaction_date'] - real_case['death_date']).days
    }
    print("Extracted Real Case Context:")
    print(case_context)
else:
    print("No deceased anomalies found in current dataset simulation.")


Extracted Real Case Context:
{'flag_type': 'DECEASED_DISBURSEMENT', 'risk_score': 100, 'beneficiary_id': 'B100004', 'aadhaar': np.int64(605645347792), 'transaction_date': '2025-01-10', 'death_date': '2024-05-29', 'days_post_mortem': 226}


## 2. Generate the Prescriptive Suggestion
We pass the data context into a simulated Generative AI pipeline (which could be OpenAI API or a local Hugging Face LLM).


In [8]:
def generate_suggestion_local_llm(case_data):
    """Simulates a Generative AI or Rule-Based response."""
    
    # The prompt engineering phase
    prompt = f"""Analyze this DBT fraud alert and suggest an action for the District Officer.
Alert Type: {case_data.get('flag_type')}
Risk Score: {case_data.get('risk_score')}
Days Post Mortem: {case_data.get('days_post_mortem')}
Transaction Date: {case_data.get('transaction_date')}
Death Date: {case_data.get('death_date')}

Actionable Suggestion:"""

    # Simulate the LLM structured output
    if case_data.get('flag_type') == 'DECEASED_DISBURSEMENT':
        days = case_data.get('days_post_mortem')
        return (f"FAULT: Aadhaar {case_data.get('aadhaar')} matched State Registry. Transfer occurred {days} days post-mortem.\n"
                f"SUGGESTION: Immediately block funds. Do NOT dispatch field verifier as death is strictly confirmed by vital statistics.")
                
    return "SUGGESTION: Review manually."

# Execute
if 'case_context' in locals():
    print("\n--- Final Prescriptive AI Output ---")
    print(generate_suggestion_local_llm(case_context))



--- Final Prescriptive AI Output ---
FAULT: Aadhaar 605645347792 matched State Registry. Transfer occurred 226 days post-mortem.
SUGGESTION: Immediately block funds. Do NOT dispatch field verifier as death is strictly confirmed by vital statistics.
